# Can a seemingly random cortex self-organise?

Salt-and-pepper cortex looks a little like someone dropped an orientation map, swept up the pieces, and put them back at random. But what if the map is not missing? What if it is simply hiding?

The feature we follow is **orientation preference**: the edge angle that makes a V1 neuron respond most strongly. In a smooth orientation map, nearby neurons prefer similar angles; in a salt-and-pepper map, different preferences appear intermixed at cellular scale.

**Self-organisation** means that nobody supplies the finished map—not a supervisor, a label, or a built-in cortical blueprint. Each neuron responds to its own input, interacts with other neurons, and adjusts its connections using local activity. Repeated many times, those small local steps can create population-level structure by themselves.

**The short answer is: yes!** In this demo, self-organisation builds a fabric of tiny, interconnected cortical domains. The structure is obvious on a perfect lattice; move the neurons only a little, and the map appears to vanish—even though the learned representation is still there. The map has not disappeared. It has gone undercover.

We will follow the clues one at a time. If you just want the story, keep scrolling; if you want to look under the bonnet, open the **Technical details** blocks.

<details>
<summary><strong>Technical details</strong></summary>

This notebook is a complete training-and-presentation workflow for micro-GCAL. If the completed archive is absent, it trains a 100×100 V1 sheet for two epochs; otherwise it loads the archived run. Code cells contain configuration and public helper calls only. Collection, checkpointing, measurements, plotting, animation, synthetic probes, and displacement analyses live in `helpers/microdomain_demo.py`.

</details>


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'demo_microdomains' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from demo_microdomains.helpers.microdomain_demo import (
    animation_html,
    animate_dimensionality_learning,
    animate_map_learning,
    animate_rotating_umap_grid,
    animate_scattered_map_learning,
    animate_synthetic_learning,
    animate_weight_learning,
    animate_robustness_learning,
    ensure_github_assets,
    estimate_demo_archive_gib,
    github_demo_config,
    macaque_displacement_table,
    plot_lgn_inputs_and_statistics,
    plot_macaque_displacement_links,
    plot_macaque_displacement_summary,
    run_macaque_displacement_demo,
    train_or_load_demo,
)

## 0. First, meet our tiny cortex

We need a cortex small enough to fit inside a computer but complicated enough to surprise us. This notebook can grow one from scratch and then replay what happened. If a completed run is already waiting, we skip the several-hour origin story and jump straight to the interesting bits.

<details>
<summary><strong>Technical details</strong></summary>

The canonical configuration uses a 100×100 V1 microdomain sheet, 80×80 natural-image crops, two independently shuffled epochs, and 100 learning snapshots. `train_or_load_demo` treats a completed archive as immutable: it loads valid existing data and trains only when that archive is absent or incomplete.

</details>


In [ ]:
config = github_demo_config()
estimate_demo_archive_gib(config)

In [ ]:
archive = train_or_load_demo(config)
archive.manifest['status'], archive.n_frames

In [ ]:
assets = ensure_github_assets(archive)
assets

## 1. Meet micro-GCAL: local competition, distant cooperation

Before we feed the model any images, here is the cast of characters. Read the figure from the bottom up: a visual stimulus becomes a sparse, contrast-normalised pattern in the **LGN**, which sends feedforward drive to a recurrent sheet representing **V1 layer 4**. This is the only cortical layer in this demo, so from here on we simply call it **V1** or **cortex**.

Each V1 neuron receives four kinds of input:

- **Afferent input** from a small local patch of LGN. These plastic connections become the neuron's visual receptive field—the region and pattern in the visual input that drive it.
- **Short-range excitation (SRE)** from its nearest cortical neighbours. Nearby co-active neurons encourage one another, helping a small patch settle together.
- **Longer-range inhibition (LRI)** from a wider surrounding neighbourhood. This suppresses competing activity and stops one excited patch from swallowing the whole sheet.
- **Cross-domain excitation (CDE)** with the widest reach. Unlike a uniform excitatory halo, CDE is learned selectively: strong links grow mainly between neurons that are repeatedly active together, allowing separated but functionally related patches to cooperate.

The spatial ordering is therefore **SRE < LRI < CDE**: local cooperation, broader competition, and selective cooperation again at the longest scale. That last ingredient is the key twist. Local excitation and inhibition keep individual domains tiny; CDE couples those tiny domains into a coordinated fabric and helps their recurrent responses resist noise. Think **many small islands connected by learned bridges**, rather than one continent—or a random scattering of boats.

The learning rule is deliberately local: neurons that are active together strengthen the connections between them. Nobody tells a neuron which orientation to prefer. Selectivity and spatial organisation emerge from the repeated cycle of seeing an input, interacting, settling, and adjusting connections.

<img src="demo_assets/microdomain/micro_gcal_architecture.png" alt="Micro-GCAL architecture with LGN input, a recurrent V1 sheet, short-range excitation, longer-range inhibition, and cross-domain excitation" width="456">

<details>
<summary><strong>Technical details</strong></summary>

Micro-GCAL belongs to the GCAL/LISSOM family of self-organising cortical-map models. For each stimulus, LGN activity provides a fixed afferent drive while V1 activity is updated recurrently for 30 settling steps. At each step, a neuron's net drive combines its afferent input, positive SRE, subtractive LRI, and positive CDE, followed by thresholding and a saturating nonlinearity.

After settling, non-negative Hebbian plasticity strengthens connections between co-active pre- and postsynaptic units. CDE uses a thresholded Hebbian rule, so appreciable long-range excitation develops only between strongly co-active neurons rather than everywhere inside its radius. Incoming weights are normalised, adaptive thresholds maintain sparse activity, and slow gain control balances afferent and recurrent contributions. Repeating this settle–learn cycle over natural-image patches jointly shapes receptive fields, recurrent connectivity, and the spatial map.

In the figure, red denotes excitation, blue denotes inhibition, and the patchiness of CDE illustrates its learned selectivity. White-to-black activity maps run from zero to maximum mean activity. The dashed line identifies one example V1 neuron and the interaction fields centred on it.

</details>


## 2. Give it something to look at

Our model starts with little glimpses of the natural world. The LGN is the relay between retina and cortex; here an LGN-like stage emphasises local light–dark boundaries, normalises their contrast, and hands sparse patches of edges and texture to V1. There are no orientation labels and no hidden answer sheet—the cortex has to work out the useful structure for itself.

<details>
<summary><strong>Technical details</strong></summary>

Natural-image crops are passed through an ON-centre Laplacian-of-Gaussian filter and divisive gain control. The fixed held-out examples below provide a common reference for intensity, exact-zero sparsity, spatial power, and the number of principal components needed to explain 95% of LGN variance.

</details>

<img src="demo_assets/microdomain/lgn_inputs.png" alt="LGN inputs and statistics" width="800">


In [ ]:
plot_lgn_inputs_and_statistics(archive)

## 3. Now let the neurons negotiate

Each new input starts a brief conversation among the neurons: excite nearby partners, inhibit competitors, settle, learn, repeat. Slowly, tiny orientation domains appear. They may look busy, but they are not random—the Fourier ring exposes their preferred spacing, while the retinotopic fishnet bends locally without losing the global plot.

Here is the visual key. In the orientation map, colour is preferred edge angle, so same-coloured neighbours form a domain. A ring in Fourier space means that similar features repeat at a characteristic distance in every direction. The retinotopy panels ask *where in the image does each cortical location look?* A globally ordered but locally bent grid means that neighbouring parts of visual space are still represented nearby, despite fine-scale distortions.

<details>
<summary><strong>Technical details</strong></summary>

For each input, activity settles through short-range excitation, plastic longer-range inhibition, and selective cross-domain excitation before Hebbian and homeostatic updates. From left to right the animation shows orientation preference, its spatial Fourier power, horizontal receptive-field position, and the central quarter of the retinotopic fishnet. In the Fourier panel, black means power and white means none.

</details>

![Orientation map formation](demo_assets/microdomain/map_learning.gif)


In [ ]:
animation_html(animate_map_learning(archive))

## 4. Connections begin choosing their friends

No one paints the finished map onto the sheet. The incoming connections learn which visual patterns matter, while cross-domain excitation learns which distant patches tend to belong together. The result is not a collection of lonely dots, but a fabric of tiny domains that have started exchanging phone numbers.

Each small tile below is one neuron's incoming connection pattern. Bright, structured afferent tiles indicate selective visual receptive fields; bright patches in the CDE tiles show which more distant cortical partners have acquired strong excitatory links.

<details>
<summary><strong>Technical details</strong></summary>

The left canvas follows sampled LGN→V1 afferent weights after independent zero-centred normalisation. The right canvas follows source-centred cross-domain excitatory fields. Only complete interior crops are used, with no border padding. Both use the same perceptually uniform scale with exact zero shown in white.

</details>

![Afferent and cross-domain plasticity](demo_assets/microdomain/weight_learning.gif)


In [ ]:
animation_html(animate_weight_learning(archive))

## 5. Can it remember a face?

A colourful map is not much use if it forgets what it saw. So we show the network the same synthetic face throughout learning and ask a **decoder**—a separate readout that sees only V1 activity—to rebuild the input. The face makes progress easy to spot, while the final curve reports average fidelity across the full fixed evaluation set. Higher cosine similarity means the reconstruction more closely matches the original input.

<details>
<summary><strong>Technical details</strong></summary>

At every snapshot a concurrently trained nonlinear decoder maps settled V1 activity back to LGN space. The first three panels show the fixed face, its cortical response, and its reconstruction. The last panel is the mean input–reconstruction cosine similarity across all held-out reconstruction examples—not the face score alone. No intermediate image is interpolated.

</details>

![Tracked reconstruction fidelity](demo_assets/microdomain/synthetic_learning.gif)


In [ ]:
animation_html(animate_synthetic_learning(archive))

## 6. The population finds a shortcut

Ten thousand neurons do not need ten thousand independent opinions. As learning proceeds, groups of neurons begin varying together, and PCA reveals that shared structure. V1 can then describe the input with fewer effective degrees of freedom—a compact code, rather than a lossy shrug.

PCA looks for recurring patterns of population activity. If many neurons change together, their responses can be summarised by fewer such patterns. Here, **effective dimensionality** is simply the number of patterns needed to explain 95% of the variation: fewer dimensions mean more shared structure in the population code.

<details>
<summary><strong>Technical details</strong></summary>

The first three panels show the spatial loadings of the leading principal components through learning. The final plot tracks the number of components required to explain 95% of V1 variance and compares it with the fixed LGN value. Their ratio is a unitless estimate of relative representational dimensionality—roughly, the fraction of input degrees of freedom retained by the cortical code.

</details>

![PCA geometry and dimensionality](demo_assets/microdomain/dimensionality.gif)


In [ ]:
animation_html(animate_dimensionality_learning(archive))

## 7. Add noise and see whether it panics

A compact code still needs to keep its story straight when conditions get messy. We inject noise during the recurrent conversation and compare the result with the clean response. Selective excitation helps the network return to nearly the same answer, so the code becomes not only efficient but dependable.

The clean and noisy panels show the same cortical region responding to the same input. Their cosine similarity is 1 when the two population responses point in exactly the same direction, and falls as noise changes the code.

<details>
<summary><strong>Technical details</strong></summary>

The same fixed LGN input and the same noise realisation are used at every learning snapshot. The middle panels compare matched 20×20 windows of clean activity and activity with noise intensity 0.06. The final plot tracks the population cosine similarity between clean and noisy settled codes through training.

</details>

![Noise robustness](demo_assets/microdomain/robustness.gif)


In [ ]:
animation_html(animate_robustness_learning(archive))

## 8. Now shake the seating plan

**Experiment 1 asks a controlled question:** what changes if development moves the neurons *after* the network has learned? We leave every receptive field and orientation preference intact, then reassign neurons to nearby cortical positions with an average displacement of two lattice spacings. This is a positional perturbation, not retraining and not added response noise.

The answer is surprisingly sharp. Short-range orientation clustering survives, but the fine periodic pattern and its Fourier ring largely disappear. That combination—weak local order without an obvious global map—is the signature we wanted to compare with salt-and-pepper cortex. It is qualitatively consistent with the cellular-scale clustering measured in mouse V1 by [Ringach et al. (2016)](https://doi.org/10.1038/ncomms12270).

<details>
<summary><strong>Technical details</strong></summary>

We draw one seeded, one-to-one local permutation. Candidate moves are weighted by a two-dimensional Gaussian and its width is calibrated until the realised mean Euclidean displacement equals `scatter_displacement = 2.0` model cells. Because this is a permutation, no neuron is duplicated or deleted. The same mapping is applied to orientation preference and receptive-field-centroid identity, and it is held fixed across all learning snapshots.

The animation provides four checks on the perturbation: the displaced orientation map shows the remaining local clusters; the Fourier panel tests whether global periodicity survives; horizontal retinotopy shows that coarse visual-field order remains; and the central-quarter fishnet makes the local distortions visible. The procedure changes where learned units are read out on the sheet, while their tuning and population responses remain unchanged.

</details>

![Controlled neuronal displacement](demo_assets/microdomain/scattered_learning.gif)


In [ ]:
scatter_displacement = 2.0
animation_html(
    animate_scattered_map_learning(
        archive, mean_displacement=scatter_displacement
    )
)

## 9. Ask real cortex how far the chairs moved

**Experiment 2 asks for a biological scale.** We cannot observe developmental migration retrospectively, so we estimate how far each measured neuron lies from a smooth orientation map that could underlie the cellular data. The distance is a model-based proxy for positional scatter—not a direct measurement of a neuron's developmental journey.

The source experiment used two-photon calcium imaging of superficial V1 in two awake, fixating macaques, with two 850×850 µm fields from each animal. We use the baseline orientation measurements released by [Chen et al. (2026)](https://doi.org/10.7554/eLife.107518), available in their [public Zenodo dataset](https://doi.org/10.5281/zenodo.20053907). This dataset is especially useful here because it combines dense cellular positions with **12 tested axial orientations at 15° spacing**. The measurements are still discrete, but much less coarse than a small set of broad orientation bins; we retain all 12 orientations without re-binning them.

The top row below shows every significantly orientation-tuned cell in the three densest fields. The bottom row shows the corresponding circularly smoothed orientation fields. If an orderly map sits underneath the cellular scatter, the gap between a soma and the nearest place predicting its preferred orientation provides an estimate of how far positional disorder has separated the two.

<details>
<summary><strong>Technical details</strong></summary>

The original release contains four fields (`MA_1`, `MA_2`, `MB_1`, and `MB_2`); we select the three with the most significantly tuned cells (`MB_1`, `MB_2`, and `MA_2`). Tuning significance follows the authors' released p&lt;0.01 selection. Preferred orientation is treated as axial—0° and 180° are identical—by smoothing the complex representation `exp(2iθ)` with one shared Gaussian σ of 100 µm.

For each neuron, its own contribution is removed before rebuilding the smooth map. We then find the nearest linearly interpolated contour that predicts exactly that neuron's preferred orientation and measure the Euclidean soma-to-contour distance. All measured cells remain visible in the scatter plots. Matches above 350 µm are excluded only from the reported means and the linked example pool, preventing rare remote contour matches from dominating the estimate.

| Dataset | Included cells | Mean displacement |
|---|---:|---:|
| `MB_1` | 792 | 91.4 µm |
| `MB_2` | 729 | 82.5 µm |
| `MA_2` | 395 | 92.3 µm |

</details>

<img src="demo_assets/microdomain/macaque_displacement_summary.png" alt="Measured macaque V1 cellular orientations above circularly smoothed maps" width="900">


In [ ]:
macaque_smoothing_sigma_um = 100.0
macaque_max_displacement_um = 350.0
macaque_results = run_macaque_displacement_demo(
    smoothing_sigma_um=macaque_smoothing_sigma_um,
    max_displacement_um=macaque_max_displacement_um,
)
plot_macaque_displacement_summary(macaque_results)
macaque_displacement_table(macaque_results)

### From a population estimate to individual correspondences

The first figure answers the population-level question: *what smooth field is compatible with the measured orientations, and what displacement scale does it imply?* The second makes that calculation tangible for individual neurons. A coloured dot marks the measured soma, the black × marks the nearest same-orientation location on the leave-one-out smooth map, and the black segment is the inferred displacement.

Only 20 reproducibly sampled neurons per field are drawn so that the paths remain readable. Their endpoints are not arbitrary nearest pixels: they lie on linearly interpolated exact-orientation contours. The faint field underneath is the same σ=100 µm circular map used for the population estimate, and the title reports the mean over all included neurons rather than over the 20 examples.

This view is useful for intuition, but the caveat matters: a line does **not** trace a cell's real migration. It visualises the spatial discrepancy between a measured soma and an inferred smooth-map location with matching tuning.

<details>
<summary><strong>Technical details</strong></summary>

The example subset is selected with a fixed random seed from cells with finite matches no greater than 350 µm. Each cell remains excluded from the map used to predict its own endpoint, avoiding a trivial zero-distance match. Colours preserve the original 12 orientation measurements from the [Chen et al. dataset](https://doi.org/10.5281/zenodo.20053907); black lines and × markers encode only the derived correspondence.

</details>

<img src="demo_assets/microdomain/macaque_displacement_links.png" alt="Example soma-to-map displacement correspondences in macaque V1" width="900">


In [ ]:
macaque_link_examples = 20
plot_macaque_displacement_links(
    macaque_results, n_links=macaque_link_examples, map_alpha=0.28
)

## 10. Leave the sheet and look for the hidden shape

The cortical map now looks scrambled, so let us stop judging the network by cortical position. In response space, the order reappears: gratings, the topological model, the salt-and-pepper model, and real mouse V1 all trace similarly smooth, folded geometries. The map can disappear from the sheet while its hidden shape survives in the code.

Each dot is the activity of an entire population for one stimulus or trial. UMAP places dots nearby when those high-dimensional activity patterns are similar, and colour marks grating orientation. A smooth colour progression around the folded shape therefore means that nearby orientations still evoke systematically related population responses—even when preferred orientations look shuffled across the cortical sheet.

Read the panels from left to right as a ladder of comparisons: the geometry already present in the grating stimuli, the responses of the topographic simulation, the responses of the salt-and-pepper simulation, and recorded mouse V1 activity.

The biological panel uses high-arousal trials from random-phase recording 1 in the public [Stringer–Pachitariu dataset](https://doi.org/10.25378/janelia.8279387.v3), associated with [Stringer et al. (2019)](https://doi.org/10.1038/s41586-019-1346-5).

<details>
<summary><strong>Technical details</strong></summary>

The four panels show separately fitted 3D UMAPs of raw full-phase grating pixels, topographic-model responses, salt-and-pepper-model responses, and 1,916 high-arousal trials from Stringer random-phase recording 1. Colour denotes orientation modulo 180°. The stimulus and model embeddings are compatible with the twisted angle–phase identification of a Klein bottle; the mouse data lack matched phase labels, so their resemblance is qualitative rather than a formal topological test. Synchronized rotation exposes the full geometry without distorting the fitted axis scales. Marker colours are fixed across frames; camera rotation changes the viewpoint, not the colour mapping.

</details>

<img src="demo_assets/microdomain/rotating_umap.gif" alt="Rotating four-panel 3D UMAP comparison" width="1050">


In [ ]:
umap_rotation_frames = 48
umap_animation_asset = assets['rotating_umap']

## Conclusion: not random—just undercover

We started with a cortex that looked as though its map had been lost. Instead, the model offers a different ending: self-organisation can build selective receptive fields, efficient codes, robust dynamics, and structured coupling at a very fine scale. Modest neuronal displacement then hides the tidy layout without undoing the learning.

So salt-and-pepper need not mean **structureless**. It may mean **beautifully organised, then very lightly shuffled**.

<details>
<summary><strong>Technical takeaway</strong></summary>

Micro-GCAL extends classical recurrent self-organisation with learned selective cross-domain excitation. On the ideal sheet it produces small coupled domains, smooth retinotopy, reduced effective dimensionality, improving reconstruction fidelity, and robust settled codes. Local positional disorder masks the high-frequency spatial signatures, while local tuning similarity and smooth population-response geometry remain measurable. This is a model-based hypothesis—not proof that every biological salt-and-pepper map develops this way—but it gives the hypothesis concrete, testable signatures.

</details>
